In [68]:
import pandas as pd
import numpy as np
from PIL import Image
pd.set_option('display.max_columns', 500)

In [69]:
chestx = pd.read_csv("../reports/chestx/layer/predictions_per_layer_test.csv")
padchest = pd.read_csv("../reports/padchest/layer/predictions_per_layer_test.csv")
padchest_px = pd.read_csv("../reports/padchest_px/layer/predictions_per_layer_test.csv")

# ChestX-ray14 (chestx)

In [70]:
chestx["prob"] = chestx["layer_17"]
chestx["dist_from_05"] = (chestx["prob"] - 0.5).abs()

cols = ["img_id", "y_true", "drain", "sex", "prob"]

no_disease = chestx.nsmallest(20, "prob")[cols].round({"prob": 5}) # closest to 0 = confident no pneumothorax
disease = chestx.nlargest(20, "prob")[cols].round({"prob": 5}) # closest to 1 = confident pneumothorax
uncertain = chestx.nsmallest(20, "dist_from_05")[cols].round({"prob": 5}) # closest to 0.5 = uncertain

print("most confident: NO pneumothorax (prob. closest to 0)")
print(no_disease)

print("most confident: pneumothorax (prob. closest to 1)")
print(disease)

print("least confident (prob. closest to 0.5)")
print(uncertain)

# hpc commands
# no disease
# scp XXX MedCLIP_subgroup/reports/chestx/confidence_images/no_disease

# disease
# scp XXX MedCLIP_subgroup/reports/chestx/confidence_images/disease

# uncertain
# scp XXX MedCLIP_subgroup/reports/chestx/confidence_images/uncertain

most confident: NO pneumothorax (prob. closest to 0)
                 img_id  y_true  drain sex     prob
6366   00010007_121.png       0      1   M  0.03302
17897  00020945_039.png       0      0   M  0.07243
25220  00030450_001.png       0      1   M  0.07381
9721   00013377_010.png       0      0   F  0.07439
25221  00030450_002.png       0      1   M  0.08059
14575  00017318_015.png       0      0   M  0.08154
9522   00013256_005.png       0      0   F  0.08522
14664  00017504_010.png       0      1   F  0.08524
19902  00023176_019.png       0      1   M  0.08804
13260  00016184_004.png       0      1   F  0.09618
902    00001247_006.png       0      0   M  0.09799
12127  00014996_005.png       0      0   M  0.09874
12130  00014996_008.png       0      1   M  0.09890
25280  00030522_001.png       0      1   M  0.09998
17557  00020482_003.png       0      0   F  0.10307
12627  00015530_156.png       0      0   M  0.10419
18359  00021420_004.png       0      0   M  0.10625
11913  0001

In [71]:
# checking the fully grey image to see if it was fully uniform (it wasn't)
image_path = "../reports/chestx/confidence_images/no_disease/00010007_121.png"
image = np.array(Image.open(image_path))

unique_vals = np.unique(image)

print(f"shape: {image.shape}")
print(f"number of unique pixel values: {len(unique_vals)}")
print(f"unique values: {unique_vals}")

shape: (1024, 1024)
number of unique pixel values: 80
unique values: [ 61  62  91  93  98  99 100 101 102 103 104 105 106 109 112 113 114 115
 116 117 118 119 120 121 122 123 140 149 151 153 159 160 163 165 167 168
 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186
 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204
 205 206 207 208 209 211 215 216]


# PadChest (cardiomegaly)

In [72]:
padchest.head()

,img_id,y_true,scanner,sex,layer_1,layer_2,layer_3,layer_4,layer_5,layer_6,layer_7,layer_8,layer_9,layer_10,layer_11,layer_12,layer_13,layer_14,layer_15,layer_16,layer_17
0,313903302629300007485735352869488750471_75sg0k...,0,PhilipsMedicalSystems,M,0.459714,0.494188,0.422652,0.479810,0.532964,0.474332,0.500394,0.305093,0.269137,0.311276,0.383264,0.337885,0.362479,0.246250,0.230337,0.266138,0.227102
1,238285621348398466668514178112618553012_a7k6dv...,0,PhilipsMedicalSystems,F,0.557211,0.584888,0.516900,0.636813,0.676344,0.611350,0.600950,0.368163,0.300543,0.315397,0.331582,0.295506,0.318948,0.245141,0.220122,0.157579,0.139315
2,152191969602076825998375638267191596461_ck9qkz...,0,ImagingDynamicsCompanyLtd,F,0.540119,0.618165,0.563688,0.731793,0.759385,0.722752,0.691495,0.597966,0.525462,0.528509,0.546008,0.530870,0.535693,0.579941,0.430067,0.321881,0.287729
3,14968019924555248865694512726537711769_3r162a.png,0,PhilipsMedicalSystems,F,0.551285,0.580406,0.501584,0.672665,0.700094,0.648869,0.626308,0.385271,0.370172,0.391201,0.450521,0.373388,0.382513,0.310637,0.306815,0.321619,0.268475
4,120005774206062253919592068222866365316_5rzk13...,0,PhilipsMedicalSystems,M,0.529326,0.567326,0.505725,0.631629,0.675707,0.620714,0.613830,0.438055,0.377414,0.398307,0.404945,0.367381,0.405711,0.254240,0.172444,0.174936,0.155133


In [73]:
idc = padchest_px[padchest_px["scanner"] == "ImagingDynamicsCompanyLtd"]
idc["y_true"].value_counts()

y_true
0    10966
1        4
Name: count, dtype: int64

In [74]:
idc = padchest_px[padchest_px["scanner"] == "PhilipsMedicalSystems"]
idc["y_true"].value_counts()

y_true
0    10722
1       78
Name: count, dtype: int64

In [75]:
padchest["prob"] = padchest["layer_17"]
padchest["dist_from_05"] = (padchest["prob"] - 0.5).abs()

cols = ["img_id", "y_true", "scanner", "sex", "prob"]

no_disease = padchest.nsmallest(20, "prob")[cols].round({"prob": 5}) # closest to 0 = confident no cardiomegaly
disease = padchest.nlargest(20, "prob")[cols].round({"prob": 5}) # closest to 1 = confident cardiomegaly
uncertain = padchest.nsmallest(20, "dist_from_05")[cols].round({"prob": 5}) # closest to 0.5 = uncertain

print("most confident: NO cardiomegaly (prob. closest to 0)")
print(no_disease.to_string())

print("most confident: cardiomegaly (prob. closest to 1)")
print(disease.to_string())

print("least confident (prob. closest to 0.5)")
print(uncertain.to_string())

# no disease
# scp XXX MedCLIP_subgroup/reports/padchest/confidence_images/no_disease

# disease
# scp XXX MedCLIP_subgroup/reports/padchest/confidence_images/disease

# uncertain
# scp XXX MedCLIP_subgroup/reports/padchest/confidence_images/uncertain

most confident: NO cardiomegaly (prob. closest to 0)
                                                             img_id  y_true                    scanner sex     prob
7100   216840111366964013686042548532013156114347330_02-105-039.png       0  ImagingDynamicsCompanyLtd   F  0.02615
274              103693917093872830847809526276065588596_q5hqnk.png       0  ImagingDynamicsCompanyLtd   F  0.02623
16238  216840111366964012959786098432011039154446686_00-178-047.png       0  ImagingDynamicsCompanyLtd   F  0.02825
16501  216840111366964012487858717522009210154307867_00-030-109.png       0      PhilipsMedicalSystems   F  0.02854
8684   216840111366964013515091760022012320082235519_01-151-073.png       0  ImagingDynamicsCompanyLtd   F  0.02898
14379  216840111366964012734950068292010145101156547_03-149-111.png       0  ImagingDynamicsCompanyLtd   F  0.02977
20271  216840111366964012558082906712009328132833379_00-102-030.png       0  ImagingDynamicsCompanyLtd   F  0.02980
15484  216840111366

# PadChest (pneumothorax)

In [79]:
padchest_px["prob"] = padchest_px["layer_17"]
padchest_px["dist_from_05"] = (padchest_px["prob"] - 0.5).abs()

cols = ["img_id", "y_true", "scanner", "sex", "prob"]

no_disease = padchest_px.nsmallest(20, "prob")[cols].round({"prob": 5}) # closest to 0 = confident no pneumothorax
disease = padchest_px.nlargest(20, "prob")[cols].round({"prob": 5}) # closest to 1 = confident pneumothorax
uncertain = padchest_px.nsmallest(20, "dist_from_05")[cols].round({"prob": 5}) # closest to 0.5 = uncertain
# uncertain = (padchest_px.nsmallest(20, "dist_from_05").sort_values("prob")[cols].round({"prob": 5}))

print("most confident: NO pneumothorax (prob. closest to 0)")
print(no_disease.to_string())

print("most confident: pneumothorax (prob. closest to 1)")
print(disease.to_string())

print("least confident (prob. closest to 0.5)")
print(uncertain.to_string())

# no disease
# scp XXX MedCLIP_subgroup/reports/padchest_px/confidence_images/no_disease

# disease
# scp XXX MedCLIP_subgroup/reports/padchest_px/confidence_images/disease

# uncertain
# scp XXX MedCLIP_subgroup/reports/padchest_px/confidence_images/uncertain

most confident: NO pneumothorax (prob. closest to 0)
                                                             img_id  y_true                    scanner sex     prob
17545  216840111366964012558082906712009322124844600_00-107-093.png       0  ImagingDynamicsCompanyLtd   F  0.04158
18735  216840111366964012558082906712009344110636318_00-089-099.png       0  ImagingDynamicsCompanyLtd   F  0.04271
4243             248261555152019367507591328512148732003_yot61b.png       0  ImagingDynamicsCompanyLtd   M  0.04431
114              147036015370488755117195440122198442935_gj6nyv.png       0  ImagingDynamicsCompanyLtd   F  0.04608
1841             208990610239313623398510691329973957149_ttvss6.png       0  ImagingDynamicsCompanyLtd   M  0.04898
11420  216840111366964013217898866992012020113614271_01-043-003.png       0  ImagingDynamicsCompanyLtd   F  0.04933
17547  216840111366964012558082906712009321142754309_00-107-079.png       0  ImagingDynamicsCompanyLtd   M  0.04963
6346   216840111366

# Bins

In [77]:
bins = [0, 0.5, 0.8, 0.9, 0.95, 0.99, 1.01]
print(chestx["prob"].value_counts(bins=bins, sort=False))

(-0.001, 0.5]     7163
(0.5, 0.8]       10381
(0.8, 0.9]        3350
(0.9, 0.95]       2110
(0.95, 0.99]      2088
(0.99, 1.01]       504
Name: count, dtype: int64
